# Multi-Asset Forecasting

This notebook studies cross-asset relationships and forecasts the target asset with help from the broader asset set.

The default universe uses tradable proxies:

- `GLD` for gold exposure
- `BNO` for Brent oil exposure
- `NVDA` for NVIDIA
- `SPY` for S&P 500 exposure

You can swap these symbols for any other assets that your TwelveData setup supports.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "src").exists() and (candidate / "backend").exists():
        repo_root = candidate
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from src.market_forecasting import (
    AssetDefinition,
    compute_lead_lag_relationships,
    run_multi_asset_experiment,
    summarize_experiment_results,
)

In [ ]:
ASSETS = {
    "GLD": AssetDefinition(symbol="GLD", label="Gold proxy via GLD ETF"),
    "BNO": AssetDefinition(symbol="BNO", label="Brent proxy via BNO ETF"),
    "NVDA": AssetDefinition(symbol="NVDA", label="NVIDIA"),
    "SPY": AssetDefinition(symbol="SPY", label="S&P 500 proxy via SPY ETF"),
}
TARGET_SYMBOL = "NVDA"
START_DATE = "2015-01-01"
HORIZONS = (1, 21)
LOOKBACK = 60
EPOCHS = 35

results, asset_frames = run_multi_asset_experiment(
    assets=ASSETS,
    target_symbol=TARGET_SYMBOL,
    start_date=START_DATE,
    horizons=HORIZONS,
    lookback=LOOKBACK,
    epochs=EPOCHS,
)

summary = summarize_experiment_results(results)
summary

In [ ]:
lead_lag = compute_lead_lag_relationships(
    asset_frames,
    target_symbol=TARGET_SYMBOL,
    horizon=1,
    max_lag=5,
)
lead_lag.head(15)

In [ ]:
returns = pd.concat(
    {symbol: np.log(frame["Close"]).diff() for symbol, frame in asset_frames.items()},
    axis=1,
).dropna()
returns.corr().round(3)

In [ ]:
latest_forecasts = pd.DataFrame(
    [
        {"horizon_days": horizon, "model": "technical", **payload["technical_forecast"]}
        for horizon, payload in results.items()
    ]
    + [
        {"horizon_days": horizon, "model": "deep", **payload["deep_forecast"]}
        for horizon, payload in results.items()
    ]
)
latest_forecasts.sort_values(["horizon_days", "model"]).reset_index(drop=True)

In [ ]:
daily_backtest_tail = results[1]["backtest"].tail(10)
monthly_backtest_tail = results[21]["backtest"].tail(10)

daily_backtest_tail, monthly_backtest_tail